In [1]:
import json

file = json.load(open("dataset_converted.json", "r"))
print(file[0])

{'messages': [{'role': 'user', 'content': 'What is the course code and credits for Digital Electronics and Logic Design?'}, {'role': 'assistant', 'content': '{"course_name": "Digital Electronics and Logic Design", "course_code": "EC-211", "course_type": "Core", "credits": 3, "contact_hours_per_week": "3L", "semester": 3}'}]}


In [2]:
!pip install unsloth trl peft accelerate bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.6/401.6 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 63.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9

In [3]:
# For GPU check
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

CUDA available: True
GPU: Tesla T4


In [15]:
from unsloth import FastLanguageModel
import torch

model_name = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"

max_seq_length = 2048
dtype = None


model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.3.8: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-bnb-4bit as a legacy tokenizer.


In [16]:
from datasets import Dataset

def format_prompt(example):
    return tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

formatted_data = [format_prompt(item) for item in file]
dataset = Dataset.from_dict({"text": formatted_data})
print(dataset[0]["text"])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 20 Mar 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

What is the course code and credits for Digital Electronics and Logic Design?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{"course_name": "Digital Electronics and Logic Design", "course_code": "EC-211", "course_type": "Core", "credits": 3, "contact_hours_per_week": "3L", "semester": 3}<|eot_id|>


In [17]:

model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=128,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

Unsloth 2026.3.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [18]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        save_strategy="epoch",
        save_total_limit=2,
        dataloader_pin_memory=False,
        report_to="none", # Disable Weights & Biases logging
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/49 [00:00<?, ? examples/s]

In [19]:


trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 49 | Num Epochs = 3 | Total steps = 21
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 97,255,424 of 3,310,005,248 (2.94% trained)


Step,Training Loss


In [21]:

from unsloth import FastLanguageModel
import os
import torch

checkpoints = [
    os.path.join('outputs', d) for d in os.listdir('outputs')
    if d.startswith('checkpoint-')
]
latest_checkpoint = max(checkpoints, key=lambda x: int(x.split('-')[-1]))
print(f"Loading checkpoint: {latest_checkpoint}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=latest_checkpoint,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=True,
)

messages = [
    {"role": "system", "content": "You are a helpful college assistant. Answer questions in a friendly, conversational way using the knowledge you have about the curriculum."},
    {"role": "user", "content": "What subjects are in semester 3?"}
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
input_ids = tokenizer(inputs, return_tensors="pt").input_ids.to("cuda")

with torch.no_grad():
    outputs = model.generate(
        input_ids=input_ids,
        max_new_tokens=256,
        use_cache=False,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
    )

input_len = input_ids.shape[1]
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
print(response)

Loading checkpoint: outputs/checkpoint-21
==((====))==  Unsloth 2026.3.8: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-bnb-4bit as a legacy tokenizer.
Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


{"semester": 3, "subjects": [{"code": "EC-321", "name": "Communication Systems", "credits": 3}, {"code": "EC-322", "name": "Linear Integrated Circuits", "credits": 3}, {"code": "EC-323", "name": "VLSI Technology", "credits": 3}, {"code": "EC-324", "name": "Antenna and Microwave Systems", "credits": 3}, {"code": "EC-325", "name": "Analog Electronics and Instrumentation", "credits": 2}, {"code": "EC-326", "name": "Computer Architecture and Organization", "credits": 3}, {"code": "EC-327", "name": "VLSI Interconnects and Packaging", "credits": 2}, {"code": "EC-371", "name": "Core Disciplines Elective-III", "credits": 2}, {"code": "EC-372", "name": "Core Disciplines Elective-IV", "credits": 2}, {"code": "EC-373", "name": "Core Disciplines Elective-V", "credits": 2}, {"


In [ ]:
model.save_pretrained_gguf("gguf_model", tokenizer, quantization_method="q4_k_m")

In [ ]:
from google.colab import files
import os

gguf_files = [f for f in os.listdir("gguf_model") if f.endswith(".gguf")]
if gguf_files:
    gguf_file = os.path.join("gguf_model", gguf_files[0])
    print(f"Downloading: {gguf_file}")
    files.download(gguf_file)

In [ ]:
################# FOR NATURAL SOUNDING


# import json

# data = json.load(open("dataset_converted.json"))

# def to_natural_language(item):
#     user = item["messages"][0]["content"]
#     output = json.loads(item["messages"][1]["content"])

#     # write a natural language version based on the output keys
#     if "subjects" in output:
#         lines = [f"- {s['name']} ({s['code']})" for s in output["subjects"]]
#         answer = f"In semester {output['semester']}, the subjects are:\n" + "\n".join(lines)
#     elif "course_name" in output:
#         answer = f"{output['course_name']} has code {output['course_code']} and is worth {output['credits']} credits."
#     else:
#         answer = str(output)  # fallback

#     return {
#         "messages": [
#             {"role": "system", "content": "You are a helpful college curriculum assistant. Answer conversationally."},
#             {"role": "user", "content": user},
#             {"role": "assistant", "content": answer}
#         ]
#     }

# new_data = [to_natural_language(item) for item in data]
# json.dump(new_data, open("dataset_natural.json", "w"), indent=2)